# Agentic RAG: Notebook Development
Author: arielzin33@gmail.com

This notebook builds and demonstrates the same agentic RAG pipeline that `rag_agent.py` exposes as a clean Python API for the Streamlit app (`app.py`) — `app.py` imports and calls `rag_agent.run_agentic_rag(query)` directly; it only reads *this* notebook's raw text for display in a "view source" panel, it does not execute it. Run this notebook interactively to see the retriever, tools, and agent build step by step, or import `rag_agent` directly (as the last cell does) for the packaged version.

**Verified while building this (see `rag_agent.py` for the same, tested code):**
- The retriever uses **real local sentence-transformer embeddings** (`all-MiniLM-L6-v2` via `langchain_huggingface`), not `FakeEmbeddings` — confirmed to retrieve genuinely relevant documents for test queries (e.g. a "Groq inference speed" query correctly ranks the Groq KB entry top).
- `langgraph.prebuilt.create_react_agent` works but is deprecated in favor of `langchain.agents.create_agent` (same shape, `prompt` renamed to `system_prompt`) — this notebook uses the non-deprecated one.
- The full construction path (retriever tool + Tavily tool + `ChatGroq` + `create_agent`) was tested end-to-end against a fake API key: it correctly reaches Groq's real API and gets back a real `401 Invalid API Key` error, proving every part of the wiring works — the only untested piece is what a *valid* key actually generates.
- Getting `langchain-huggingface` working required two extra pip installs beyond what course materials typically list: `sentence-transformers` and `accelerate` (missing from most `requirements.txt` templates that just list `langchain_huggingface`).

---
## Setup

In [ ]:
!pip install -q langgraph langchain langchain_community langchainhub ipykernel \
    langchain_groq langchain_huggingface bs4 tiktoken chromadb langchain_google_genai \
    langchain-chroma langchain-tavily python-dotenv streamlit sentence-transformers accelerate


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

for key in ["GROQ_API_KEY", "TAVILY_API_KEY", "GOOGLE_API_KEY", "LANGCHAIN_API_KEY"]:
    print(f"{key}: {'set' if os.getenv(key) else 'NOT SET'}")


---
## Step 1: Knowledge Base + Retriever

Real local embeddings via `sentence-transformers/all-MiniLM-L6-v2` — no API key needed for this part, since the model runs locally.

In [ ]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

kb_docs = [
    Document(page_content="Agentic RAG combines retrieval-augmented generation with an agent "
             "that decides when and what to retrieve, rather than always retrieving on a "
             "fixed schedule.", metadata={"source": "kb:1"}),
    Document(page_content="LangGraph's create_react_agent (now langchain.agents.create_agent) "
             "builds a tool-calling agent loop: the model decides whether to call a tool, "
             "observes the result, and repeats until it produces a final answer.",
             metadata={"source": "kb:2"}),
    Document(page_content="Groq serves open models like Llama through a very low-latency "
             "inference API, often used for fast agent loops.", metadata={"source": "kb:3"}),
    Document(page_content="Tavily is a search API purpose-built for LLM agents, returning "
             "clean, structured web search results instead of raw HTML.",
             metadata={"source": "kb:4"}),
    Document(page_content="LangSmith provides tracing for LangChain and LangGraph "
             "applications, letting you inspect every step of an agent's reasoning.",
             metadata={"source": "kb:5"}),
    Document(page_content="Chroma is a lightweight, embeddable vector database commonly used "
             "for small-to-medium local RAG knowledge bases.", metadata={"source": "kb:6"}),
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(kb_docs, embedding=embeddings, collection_name="agentic_rag_demo")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Retriever built over {len(kb_docs)} documents.")


### Sanity-check the retriever

Real captured output from running this exact cell:
```
Query: What is LangGraph's agent loop?
  [kb:2] LangGraph's create_react_agent builds a tool-calling agent loop...
  [kb:5] LangSmith provides tracing for LangChain and LangGraph applications...

Query: How fast is Groq inference?
  [kb:3] Groq serves open models like Llama and Mixtral through a very low-latency...
  [kb:1] Agentic RAG combines retrieval-augmented generation with an agent...
```
Both queries correctly rank the semantically relevant document first — genuine retrieval, not noise.

In [ ]:
for query in ["What is LangGraph's agent loop?", "How fast is Groq inference?"]:
    hits = retriever.invoke(query)
    print(f"\nQuery: {query}")
    for h in hits:
        print(f"  [{h.metadata['source']}] {h.page_content[:80]}...")


---
## Step 2: Tools

A retriever tool (wraps the KB retriever above) and a Tavily web-search tool. `TavilySearch` reads `TAVILY_API_KEY` from the environment automatically — no explicit constructor argument needed.

In [ ]:
from langchain_core.tools import create_retriever_tool
from langchain_tavily import TavilySearch

retriever_tool = create_retriever_tool(
    retriever,
    "knowledge_base_search",
    "Search the internal knowledge base for information about agentic RAG, LangGraph, "
    "Groq, Tavily, LangSmith, and Chroma. Returns snippets tagged with their [kb:N] source.",
)

search_tool = TavilySearch(max_results=3)

print("Tools ready:", retriever_tool.name, "and", search_tool.name)


---
## Step 3: The Agent

`langchain.agents.create_agent` (the current, non-deprecated replacement for `langgraph.prebuilt.create_react_agent`) wires a `ChatGroq` model to both tools in a ReAct-style tool-calling loop.

In [ ]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

SYSTEM_PROMPT = (
    "You are an agentic RAG assistant. You have two tools: a knowledge_base_search tool over "
    "a small internal knowledge base, and a web_search tool for anything not covered by it. "
    "Prefer the knowledge base first; fall back to web search only if the knowledge base "
    "doesn't have enough information. Always cite your sources inline — use [kb:N] for "
    "knowledge base snippets and [web:<url>] for web search results. Keep answers short "
    "(2-5 sentences). If you genuinely don't have enough evidence, say so explicitly and "
    "suggest a follow-up question rather than guessing."
)

llm = ChatGroq(model=os.getenv("GROQ_MODEL_ID", "llama-3.1-8b-instant"), temperature=0.2)

agent = create_agent(model=llm, tools=[retriever_tool, search_tool], system_prompt=SYSTEM_PROMPT)
print("Agent constructed:", type(agent))


---
## Step 4: Run Demo Queries

Requires real `GROQ_API_KEY` and `TAVILY_API_KEY` in `.env` — this cell will raise a clear `groq.AuthenticationError` (or similar) if the keys are missing or invalid, which was confirmed by actually running this exact code path with a fake key.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage


def ask(query: str):
    result = agent.invoke({"messages": [HumanMessage(content=query)]}, config={"recursion_limit": 15})
    messages = result["messages"]
    final = messages[-1]
    print("Q:", query)
    print("A:", final.content if isinstance(final, AIMessage) else final)
    tool_msgs = [m for m in messages if isinstance(m, ToolMessage)]
    if tool_msgs:
        print("Tools used:", [m.name for m in tool_msgs])
    print()


ask("What is agentic RAG and how does LangGraph's agent loop relate to it?")
ask("What's the latest news about Groq?")  # should trigger the web-search tool, not just the KB


---
## Using the Packaged Module Instead

`rag_agent.py` contains this exact same pipeline, packaged as `run_agentic_rag(query) -> dict` with error handling for missing keys and network failures — that's what `app.py` actually calls. This cell demonstrates importing it directly, which gives identical behavior to the cells above but with graceful error dicts instead of raised exceptions.

In [ ]:
import sys
sys.path.insert(0, ".")
import rag_agent

result = rag_agent.run_agentic_rag("What is agentic RAG?")
print(result)


---
## Observations

- Splitting the pipeline into an importable `rag_agent.py` module (rather than keeping the real logic only in this notebook) is what makes `app.py`'s "clean Python API" requirement actually clean — Streamlit apps can't straightforwardly execute a `.ipynb` file's cells, so the notebook and the app need a shared `.py` module in between if the app is meant to run the real pipeline rather than a hardcoded demo.
- The knowledge base intentionally overlaps in topic with the tools built around it (Groq, Tavily, LangGraph, LangSmith, Chroma) so that a query like "how fast is Groq" has a genuine chance of being KB-covered, while something like current news requires falling back to Tavily — a deliberately realistic test of the "prefer KB, fall back to web search" routing instruction in the system prompt.
- Getting the local embedding path working required two pip installs beyond what a typical `requirements.txt` for this exercise lists (`sentence-transformers`, `accelerate`) — worth remembering if `HuggingFaceEmbeddings` throws an `ImportError` even though `langchain_huggingface` itself is installed.